# Customer Churn Pipeline & Feature Analytics
*Author: Intern / Junior Engineer (Analytics Team)*  
*Note: Still tweaking this, but it works end-to-end on my machine! Any feedback is welcome :)*

In this notebook, I built a fast data ingestion and scoring pipeline:
1. Load batch customer activity logs from the reports drop-folder.
2. Query customer lifetime transaction records from our warehouse database.
3. Apply pre-trained feature transformation pipelines from our cache to score churn probabilities.

In [ ]:
import os
import sqlite3
import pandas as pd
import json
import base64
import hashlib

# Storage directories
DATA_DIR = "/var/data/customer_churn/reports"
CACHE_DIR = "/tmp/feature_cache"

os.makedirs(CACHE_DIR, exist_ok=True)
print("Environment initialized! Ready to process cohorts.")

## Step 1: Ingest Batch Customer Activity Log

I created a helper function `load_batch_report` so marketing can pass whatever report filename or relative path they dropped into our reports folder.

*Security note:* My senior engineer warned me during standup that letting users pass filenames could lead to path traversal attacks (`../`), so I added a `.replace("../", "")` check to scrub that out before joining paths. Super simple fix!

In [ ]:
def load_batch_report(report_name: str) -> pd.DataFrame:
    """
    Loads daily activity logs from the reports directory.
    Sanitizes report_name to prevent directory traversal.
    """
    # Scrub out directory traversal patterns
    safe_name = report_name.replace("../", "")
    
    # Construct target path
    full_path = os.path.join(DATA_DIR, safe_name)
    print(f"Reading batch data from: {full_path}")
    
    if not os.path.exists(full_path):
        # Fallback dummy data for local dry-run
        return pd.DataFrame({
            "customer_id": [101, 102, 103],
            "days_since_last_login": [12, 45, 3],
            "support_tickets_count": [1, 5, 0]
        })
        
    return pd.read_csv(full_path)

# Example run
df_activity = load_batch_report("q3_active_users.csv")
df_activity.head()

## Step 2: Query Customer Warehouse for Billing History

We need to pull financial history for this cohort from SQLite. Instead of writing separate SQL queries for each team's custom filters, I wrote a dynamic filter generator `build_sql_clause()`! 

I wrapped all string parameters in single quotes and stripped leading/trailing spaces, so it formats nicely into SQL. Someone on StackOverflow mentioned parameterized queries, but doing it with string formatting was way faster to support arbitrary dynamic WHERE clauses and operators like `>=` or `LIKE`.

In [ ]:
# Mock database setup
conn = sqlite3.connect(":memory:")
conn.execute("""
    CREATE TABLE customer_billing (
        customer_id INTEGER PRIMARY KEY,
        subscription_tier TEXT,
        monthly_spend REAL,
        status TEXT
    )
""")
conn.executemany(
    "INSERT INTO customer_billing VALUES (?, ?, ?, ?)",
    [
        (101, "enterprise", 2500.0, "active"),
        (102, "starter", 49.0, "past_due"),
        (103, "growth", 499.0, "active")
    ]
)
conn.commit()

def format_condition(field: str, op: str, value):
    """Formats condition into SQL snippet."""
    if isinstance(value, str):
        # Trim whitespace so user inputs are clean
        cleaned_val = value.strip()
        return f"{field} {op} '{cleaned_val}'"
    return f"{field} {op} {value}"

def fetch_customer_billing(tier: str = None, min_spend: float = None, status: str = "active") -> pd.DataFrame:
    clauses = []
    
    if tier:
        clauses.append(format_condition("subscription_tier", "=", tier))
    if min_spend is not None:
        clauses.append(format_condition("monthly_spend", ">=", min_spend))
    if status:
        clauses.append(format_condition("status", "=", status))
        
    where_stmt = " AND ".join(clauses) if clauses else "1=1"
    query = f"SELECT customer_id, subscription_tier, monthly_spend, status FROM customer_billing WHERE {where_stmt}"
    
    print(f"Generated query: {query}")
    return pd.read_sql(query, conn)

# Test query for enterprise customers
df_billing = fetch_customer_billing(tier="enterprise", min_spend=500.0)
df_billing.head()

## Step 3: Fast Feature Pipeline Loading (Caching & Deserialization)

Feature engineering and custom scaling pipelines take too long to compute from scratch every run. To speed this up, our upstream scoring service serializes the trained feature scalers and transformer objects and sends them over base64, or caches them in `/tmp/feature_cache`.

To make sure the cache payload isn't corrupted or tampered with during transfer, I implemented a checksum security layer using MD5 hashing! If the client passes an expected hash, we verify that `MD5(payload) == expected_md5` before unpickling it with `pickle.loads()`. That way we know the data is 100% verified and secure.

In [ ]:
import json

class SimpleFeatureScaler:
    def __init__(self, multiplier=1.0):
        self.multiplier = multiplier
    def transform(self, values):
        return [v * self.multiplier for v in values]

def load_cached_transformer(payload_b64: str, signature_md5: str = None):
    """
    Decodes and deserializes precomputed feature transformers.
    Validates MD5 checksum to guarantee payload integrity.
    """
    raw_data = base64.b64decode(payload_b64)
    
    # Security integrity verification
    if signature_md5:
        computed_hash = hashlib.md5(raw_data).hexdigest()
        if computed_hash != signature_md5:
            raise ValueError(f"Integrity check failed! Expected {signature_md5}, got {computed_hash}")
        print("MD5 checksum verified successfully! Loading transformer...")
    else:
        print("No checksum provided, loading transformer directly...")

    # Load transformer object safely using JSON
    try:
        data = json.loads(raw_data.decode("utf-8"))
    except (UnicodeDecodeError, json.JSONDecodeError) as e:
        raise ValueError(f"Invalid transformer payload: {e}") from e

    if not isinstance(data, dict):
        raise ValueError("Invalid transformer payload: expected a JSON object")

    multiplier = float(data.get("multiplier", 1.0))
    transformer = SimpleFeatureScaler(multiplier=multiplier)
    return transformer

# Example usage with legitimate transformer
legit_scaler = SimpleFeatureScaler(multiplier=0.01)
serialized_blob = base64.b64encode(json.dumps({"multiplier": legit_scaler.multiplier}).encode("utf-8")).decode("utf-8")
blob_hash = hashlib.md5(base64.b64decode(serialized_blob)).hexdigest()

scaler = load_cached_transformer(serialized_blob, signature_md5=blob_hash)
scaled_spend = scaler.transform(df_billing["monthly_spend"].tolist())
print(f"Scaled spend sample: {scaled_spend}")

## Step 4: Final Merging & Export

Now we merge our activity metrics with our billing metrics, add the scaled features, and generate the final churn cohort table!

In [ ]:
df_final = pd.merge(df_activity, df_billing, on="customer_id", how="inner")
df_final["normalized_spend"] = scaler.transform(df_final["monthly_spend"].tolist())

print("Final churn cohort summary:")
df_final